# Week 1 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-01.md`](../../weeks/week-01.md) · **Given engine:** [`cipher.py`](cipher.py)

Teaching walkthrough of the week-1 studio. It **imports the reference attack from
[`solution.py`](solution.py)** — never re-pasting it — so what runs here is exactly
what `test_cipher.py` grades. Correctness is verified separately:
`python3 studios/_verify_solutions.py week-01`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [1]:
# --- bootstrap: week folder (for solution/cipher) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-01"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import cipher
import solution
bank = cipher.load_ciphertexts()
CRACK_SEED = bank["crack_seed"]

## The attack, in two pieces

`frequency_guess_key` is the noisy first pass (single-letter frequency rank);
`crack` refines it by hill-climbing a **public** bigram model. Neither ever touches
the key or the plaintext — the only inputs are the ciphertext and public knowledge
of English. That constraint *is* the Kerckhoffs lesson.

In [2]:
print(inspect.getsource(solution.frequency_guess_key))
ct0 = bank["english"][0]["ciphertext"]
guess = solution.frequency_guess_key(ct0)
vals = list(guess.values())
assert len(vals) == len(set(vals)), "must be 1-to-1"
print("valid 1-to-1 frequency map over", len(vals), "symbols")

def frequency_guess_key(ciphertext):
    """Map cipher symbols to English letters by frequency RANK (commonest cipher
    symbol -> E, next -> T, ...). A noisy first pass, but readably close — and
    that "readably close" is the tell that the English assumption is right."""
    cipher_by_freq = [sym for sym, _ in letter_counts(ciphertext).most_common()]
    english_by_freq = sorted(ENGLISH_FREQ, key=ENGLISH_FREQ.get, reverse=True)
    return {sym: eng for sym, eng in zip(cipher_by_freq, english_by_freq)}

valid 1-to-1 frequency map over 23 symbols


In [3]:
print(inspect.getsource(solution.crack))

def crack(ciphertext, restarts=8, iters=3000, seed=0):
    """Recover the decryption key by random-restart hill-climbing the bigram
    ``score``. Never enumerates 26! — searches only "keys that produce English".

    Uses a seeded RNG so results are reproducible. Reads only the ciphertext and
    the PUBLIC score/ENGLISH_FREQ — never the true key or plaintext.
    """
    rng = random.Random(seed)
    symbols = list(ALPHABET)
    guess = frequency_guess_key(ciphertext)

    best_key, best_score = None, float("-inf")
    for r in range(restarts):
        # Restart 0 starts from the frequency guess (completed to a full
        # permutation); later restarts start from random permutations for the
        # diversity that lets a bigram climber escape a bad basin.
        if r == 0:
            key = dict(guess)
            used = set(key.values())
            spare = [c for c in ALPHABET if c not in used]
            rng.shuffle(spare)
            for sym in (c for c in ALPHABET if c not 

## The guarantee test — watch confidentiality collapse

A substitution cipher's confidentiality is real *only* under an unstated assumption:
that the plaintext has no exploitable structure. Make it English and the guarantee
falls — the attack recovers the plaintext **without the key**, on every one of the 12
keys. This reproduces `test_english_assumption_collapses_confidentiality`.

In [4]:
recovered_on = 0
for item in bank["english"]:
    ct, pt = item["ciphertext"], item["plaintext"]
    key = solution.crack(ct, seed=CRACK_SEED)          # ciphertext only
    rate = cipher.recovery_rate(cipher.apply_guess(ct, key), pt)
    assert rate >= 0.90, f"key_seed={item['key_seed']}: only {rate:.0%}"
    recovered_on += 1
print(f"confidentiality collapsed on {recovered_on}/{len(bank['english'])} English ciphertexts")
# show one break in the clear:
ct, pt = bank["english"][0]["ciphertext"], bank["english"][0]["plaintext"]
print("\nrecovered:", cipher.apply_guess(ct, solution.crack(ct, seed=CRACK_SEED))[:120], "...")

confidentiality collapsed on 12/12 English ciphertexts

recovered: SECURITY THROUGH OBSCURITY IS THE RELIANCE ON SECRECY OF DESIGN AS THE MAIN METHOD
OF PROVIDING SECURITY FOR A SYSTEM. A ...


## The other face — strip the assumption and the cipher holds

Encrypt a uniform-random plaintext and the *identical* attack recovers almost
nothing. The cipher is exactly as strong as the assumption is true — the algorithm
never changed. This is `test_cipher_holds_on_non_english_plaintext`.

In [5]:
item = bank["non_english"]
key = solution.crack(item["ciphertext"], seed=CRACK_SEED)
rate = cipher.recovery_rate(cipher.apply_guess(item["ciphertext"], key), item["plaintext"])
assert rate < 0.40, f"expected the cipher to hold, got {rate:.0%}"
print(f"non-English plaintext held: only {rate:.0%} recovered — the assumption breaks the cipher, not the math")

non-English plaintext held: only 0% recovered — the assumption breaks the cipher, not the math


## Defeat your own attack (the deliverable)

The studio's analysis task: encrypt so frequency analysis **fails**, then state the
defense in Control Scorecard terms.

| Defense | Bucket | Why |
|---|---|---|
| Compress/whiten the plaintext before encrypting (remove English structure) | **GUARANTEE (conditional)** | *If* the plaintext's symbol distribution is uniform, single-letter/bigram frequency analysis has nothing to bite on — shown directly above. The condition is the whole claim. |
| Keep the cipher secret ("nobody knows it's substitution") | **RAISES COST only** | Security-through-obscurity: one leaked design and the 4·10²⁶ keyspace falls to public knowledge. Kerckhoffs' point. |

**Name the assumption = name the attack.** "It's harder now" is not a guarantee;
"uniform plaintext ⇒ no frequency signal" is.